In [23]:
import pandas as pd
import os
%run utilities.ipynb
from openpyxl.styles import Alignment, Font # https://openpyxl.readthedocs.io/en/stable/styles.html
from openpyxl.styles.borders import Border, Side

In [24]:
# (1) get report data

start_time0 = time.time()
start_time  = time.time()
#print('Getting the Schedule IB report inputs ...')

# fund long name lookup
pth_nl      = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\2A - Fund Codes, Breach Register.xlsx'
nl          = pd.read_excel(pth_nl, sheet_name = 'Funds', index_col = None,  header = 0, usecols = 'A,B').dropna(subset = ['Fund Code'])
py_reports  = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
funds       = pd.read_excel(py_reports, sheet_name = 'r28ib', usecols = 'A').dropna()
fund_list   = (',').join(funds['Funds'])
date        = pd.read_excel(py_reports, sheet_name = 'r28ib', usecols = 'C', nrows = 1).iloc[0,0]
basis       = pd.read_excel(py_reports, sheet_name = 'r28ib', usecols = 'D', nrows = 1).iloc[0,0]  # indicator to show 'SYTH' or not
#fund        = funds.iloc[0,0]                                  

fund = 'PIMBAL'

print(f'Schedule IB reports as at {date.strftime("%A %d %b %Y")} for {len(funds)} fund{"" if len(funds) == 1 else "s"}: ', '\n', f'{fund_list}')
print(f'{timediff(start_time, time.time())}: getting the Schedule IB report inputs with pd.read_excel() completed')

Schedule IB reports as at Wednesday 31 Jul 2024 for 1 fund:  
 PIMBAL
2.7sec: getting the Schedule IB report inputs with pd.read_excel() completed


In [25]:
# (2) set up r28 dataframe

start_time = time.time()
#print('Setting up dataframe of values for the schedule ...')

# dataframe the Reg 28 classifications report
rpts = r'P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting'
rg   = pd.read_excel(os.path.join(rpts,f'{fund} Reg28 {date.strftime("%d%b%Y")}.xlsx'))
syth = len(rg[rg['Investment Type'] == 'SYTH'])

# remove cash contra / synthetic cash rows in-place
rg   = rg[rg['Investment Type'] != 'SYTH']

# sort reg28 by classification and issuer
# https://stackoverflow.com/questions/33165734/update-index-after-sorting-data-frame
# https://stackoverflow.com/questions/17141558/how-to-sort-a-pandas-dataframe-by-two-or-more-columns
rg = rg.sort_values(by=['Reg 28 Classification', 'Issuer', 'Primary Asset ID'], ascending = [True, False, False], ignore_index=True)

# dataframe the complete list of categories and their limits
#pth_limits = r'P:\Working Folders\Hilton\W\!Reg28_SchIB.xlsm'
pth_limits = r'P:\Working Folders\Hilton\W\!Reg28Templates.xlsx'
limits = pd.read_excel(pth_limits, sheet_name = 'Static', usecols = 'A:C', index_col = None).dropna() # complete categories
#len(limits)

# merge the Reg 28 and the limits dataframes
r28 = rg.merge(limits, left_on = 'Reg 28 Classification', right_on = 'Reg 28 Classification', how = 'left')

# add a combined instrument name column (ID + Description) 
# https://stackoverflow.com/questions/19377969/combine-two-columns-of-text-in-pandas-dataframe
r28['Instr'] = r28['Primary Asset ID'] + ' - ' + r28['i Issue Name']

# get the unique item categories in Reg 28
ctgs = r28['Reg 28 Classification'].unique()
len(ctgs)

# rename the columns of the merged dataframe
headings = {'Reg 28 Classification': 'R28C', 'End Market Value': 'EMV', 'Percentage of Market Value': 'PMV',
            'Closing Exposure PA': 'CEPA', 'Issuer Limit': 'IL', 'Aggr Limit': 'AL'}
r28      = r28.rename(columns = headings)

print('\n', f'{fund} on {date.strftime("%A %d %b %Y")}: {syth} contra{"s" if syth > 1 else ""} removed, ', 
      f'{len(ctgs)} categor{"y" if len(ctgs) == 1 else "ies"}:', '\n', (', ').join(ctgs), '\n')

print(f'{timediff(start_time, time.time())}: setting up dataframe of values for the schedule completed: ')


 PIMBAL on Wednesday 31 Jul 2024: 11 contras removed,  19 categories: 
 1.1(a), 1.1(b), 1.1(c), 1.2(a), 1.2(c), 2.1(a), 2.1(b), 2.1(d)(i), 2.1(d)(ii), 2.1(e)(i), 2.1(e)(ii), 2.2(a), 2.2(d)(i), 3.1(a)(i), 3.1(a)(ii), 3.1(a)(iii), 3.2(a)(i), 4.1(a)(i), 4.1(a)(ii) 

0.3sec: setting up dataframe of values for the schedule completed: 


In [26]:
# (3) copy the schedule template and then update static values on it

start_time = time.time()
#print('Updating static values on the schedule ...')

import openpyxl
pth_tmpl   = r'P:\Working Folders\Hilton\W\!Reg28Templates.xlsx'
wb         = openpyxl.load_workbook(pth_tmpl) # open the Reg Schedule IB template
del          wb['Static']
del          wb['Tbl2']
sh         = wb['SchIB']
wb.close # https://openpyxl.readthedocs.io/en/stable/optimized.html

# set tab name of IB sheet
sh.title   = f'{fund} SchIB {date.strftime("%d%b%Y")}'

# enter fund name and report date on IB sheet
sh['A2']   = f'{nl[nl["Fund Code"] == fund].iat[0,1]} ({fund})' # fund long name
sh['A5']   = f'As at {date.strftime("%d %B %Y")}'

# enter totals on IB sheet
sh['F8']   = r28['EMV'].sum() # TOTAL NAV
sh['F13']  = r28['EMV'].sum() # TOTAL NAV
sh['E326'] = r28['EMV'].sum() # TOTAL NAV
sh['F326'] = 100              # TOTAL %

# 3(i), 3(f), and 3(g) totals on the IB sheet
clmns = {4: 'EMV', 5: 'PMV'}
for clmn in clmns:
    # whole category number 3(f), 3(g), and 3(i) subtotals
    w3f3g = {'339': '2.1(e)(ii)', '340': '3.1(b)', '341': '4.1(b)', '348': '3.1(b)', '364': '1.2(c)'}
    for item in w3f3g:
        sh[item][clmn].value = r28[r28['R28C'] == w3f3g[item]][clmns[clmn]].sum()

    # triple category number 3(f), 3(g), and 3(i) subtotals
    t3f3g = {329: '1.2', 330: '2.2', 331: '3.2', 332: '4.2', 333: '5.2'} # three digits
    for item in t3f3g:
        sh[item][clmn].value = r28[r28['R28C'].str[:3] == t3f3g[item]][clmns[clmn]].sum()

    # double 3(f), 3(g), and 3(i) subtotals
    d3f3g = {344: '10'} # two digits
    for item in d3f3g:
        sh[item][clmn].value = r28[r28['R28C'].str[:2] == d3f3g[item]][clmns[clmn]].sum()
        
    # single 3(f), 3(g), and 3(i) subtotals
    s3f3g = {342: '8', 343: '9', 349: '9'} # one digit
    for item in s3f3g:
        sh[item][clmn].value = r28[r28['R28C'].str[:1] == s3f3g[item]][clmns[clmn]].sum()

    # hedge fund, private equity fund, and 'Other" subtotals
    sh[334][clmn].value = r28[r28['R28C'].str[:1] == '8'][clmns[clmn]].sum() + \
    r28[r28['R28C'].str[:1] == '9'][clmns[clmn]].sum() + r28[r28['R28C'].str[:2] == '10'][clmns[clmn]].sum()

# remove 'contra' line items
if basis == 'Market Value':
    contras = ['1.1(a) contra', '1.2(a) contra']
    for contra in contras:
        ct = item_row(sh, contra, 7)
        sh.delete_rows(ct, 4)
    sh.delete_rows(item_row(sh, 'Contra', 7), 1)
else:
    # foreign contra subtotals
    sh[item_row(sh, 'Contra', 7)][4].value    = r28[(r28['Investment Type'] == 'SYTH') & (r28['R28C'].str[2] == '2')]['CEPA'].sum()
    sh[item_row(sh, 'Contra', 7)][5].value    = r28[(r28['Investment Type'] == 'SYTH') & (r28['R28C'].str[2] == '2')]['CEPA'].sum() / \
                                                r28['EMV'].sum() * 100

# all subtotals
clmns = {4: 'EMV', 5: 'PMV'}
cts   = ['1', '1.1', '1.1(a)', '1.1(b)', '1.1(c)', '1.1(d)', '1.2', '1.2(a)', '1.2(b)', '1.2(c)', '2', '2.1', '2.1(a)', '2.1(b)', '2.1(c)', \
         '2.1(d)', '2.1(e)', '2.2', '2.2(a)', '2.2(b)', '2.2(c)', '2.2(d)', '2.2(e)', '3', '3.1', '3.1(a)', '3.1(b)', '3.2(a)', '3.2(b)', \
         '4', '4.1', '4.1(a)', '4.1(b)', '4.2', '4.2(a)', '4.2(b)', '5', '5.1', '5.1(a)', '5.2', '5.2(a)', '6', '6(a)', '6(b)', '7', '8', \
         '8.1', '8.1(a)', '8.2', '8.2(a)', '9', '9.1', '9.1(a)', '9.2', '9.2(a)', '10', '10.1', '10.2']
for clmn in clmns:
    for ct in cts:
        sh[item_row(sh,ct,7)][clmn].value = r28[r28['R28C'] == ct][clmns[clmn]].sum()

# 3(i) sum of subtotals
sum_foreign                             = r28[r28['R28C'].str[2] == '2']['EMV'].sum() + r28[r28['R28C'] == '10.2']['EMV'].sum()
sh[item_row(sh, '3(i)', 7)][4].value    = sum_foreign
sh[item_row(sh, '3(i)', 7)][5].value    = sum_foreign   / r28['EMV'].sum() * 100

#3(f) sum of subtotals
sum_unlisteds                           = r28[r28['R28C'] == '2.1(e)(ii)']['EMV'].sum() + r28[r28['R28C'] == '3.1(b)'    ]['EMV'].sum() + \
                                          r28[r28['R28C'] == '4.1(b)'    ]['EMV'].sum() + r28[r28['R28C'].str[:1] == '8' ]['EMV'].sum() + \
                                          r28[r28['R28C'].str[:1] == '9' ]['EMV'].sum() + r28[r28['R28C'].str[:2] == '10']['EMV'].sum()
sh[item_row(sh, '3(f)', 7)][4].value    = sum_unlisteds
sh[item_row(sh, '3(f)', 7)][5].value    = sum_unlisteds / r28['EMV'].sum() * 100

# 3(g) sum of subtotals
sum_pef                                 = r28[r28['R28C'] == '3.1(b)']['EMV'].sum() + r28[r28['R28C'].str[:1] == '9']['EMV'].sum()
sh[item_row(sh, '3(g)', 7)][4].value    = sum_pef
sh[item_row(sh, '3(g)', 7)][5].value    = sum_pef       / r28['EMV'].sum() * 100

print(f'{timediff(start_time, time.time())}: updating static values on the schedule completed')

0.3sec: updating static values on the schedule completed


In [27]:
# (4) paste instruments and then issuers onto the schedule

start_time = time.time()
#print('Pasting instruments and issuers for each of the categories ...')
# update aggregate totals for the given item - https://openpyxl.readthedocs.io/en/stable/editing_worksheets.html
#for ctg in ctgs:
ctg      = '1.1(b)'                         # set the next item / category to be populated on the sheet
k        = r28[r28['R28C'] == ctg]          # set of all securities for the item / category
k.reset_index(drop = True, inplace = True)  # Saturn Cloud How to Reset Index in a Pandas Dataframe
row_item = item_row(sh, ctg, 7)             # item row number

print(ctg, row_item, len(k))

1.1(b) 22 83


In [80]:
# paste the total MV and % MV for that category in the row of that category
clmns = {4: 'EMV', 5: 'PMV'}
for clmn in clmns:
    sh[item_row(sh, ctg, 7)][clmn].value = k[clmns[clmn]].sum()

# static row numbers
space1   = 1 # number of rows between category identifier in column 7 and first issuer in that category
space2   = 1 # number of rows between issuers' securities in a category

# delete the "No assets ..." and "Per issuer" rows then insert blanks rows on which to paste all the secuirties for the given category (= len(k))
sh.delete_rows(row_item + space1, 2)                # sh.delete_rows(sheet_row_number, number_of_rows_to_be_deleted)
sh.insert_rows(row_item + space1, len(k))           # sh.insert_rows(first row location, number of rows to insert)
for row in range(row_item, row_item + len(k)):      # left indent each of the security name cells - https://www.youtube.com/watch?v=HR9ZHgTIxzU
    sh['C'][row].alignment = Alignment(indent = 3)
    
ht       = 14.5

# paste code + description, MV and %MV for each security in the category and uniformly format security row heights
for index, row in k.iterrows():                     # loop over each security
    rw = row_item + space1 + index                  # iterate sequentially over the empty rows, pasting securities along the way ...
    sh[rw][2].value = row['Instr']
    sh[rw][4].value = row['EMV']
    sh[rw][5].value = row['PMV']
    sh[rw][4].border = Border(left  = Side(style = 'thin'))
    sh[rw][5].border = Border(right = Side(style = 'thin'))
    sh.row_dimensions[rw].height = ht # https://stackoverflow.com/questions/37891149/openpyxl-auto-height-row

# set of issuers and their sub-totals for the category - https://stackoverflow.com/questions/51971384/pandas-groupby-does-not-preserve-order
m = (k.groupby(['Issuer', 'IL'], as_index = False, sort = False)
             .agg({'CCY':'count', 'EMV': 'sum', 'PMV': 'sum'})
             .rename(columns = {'CCY':'N'})
    )

# paste code + description, issuer limit, MV and %MV for each issuer in the category
rw = row_item + space1                            # the active row for the following pastes
for idx, row in m.iterrows():
    # insert number of blank rows on which to paste all the securities (=len(m)) for the given item
    sh.insert_rows(rw, 1) # sh.insert_rows(first row location, number of rows to insert)
    sh[rw][2].value             = row['Issuer']
    sh[rw][3].value             = row['IL']
    sh[rw][4].value             = row['EMV']
    sh[rw][5].value             = row['PMV']
    sh[rw][4].border            = Border(bottom = Side(style = 'thin'))
    sh[rw][5].border            = Border(bottom = Side(style = 'thin'))
    sh[rw + row['N']][4].border = Border(bottom = Side(style = 'thin'), left  = Side(style = 'thin'))
    sh[rw + row['N']][5].border = Border(bottom = Side(style = 'thin'), right = Side(style = 'thin')) 
    sh.insert_rows(rw + row['N'] + 1,  space2)   # sh.insert_rows(first row location, number of rows to insert)   
    rw =           rw + row['N'] + 1 + space2    # row['N'] contains the number of securities associated with the issuer

#print(f'{timediff(start_time, time.time())}: pasting instruments and issuers for category {ctg} completed')
print(f'{timediff(start_time, time.time())}: pasting instruments and issuers for each of the {len(ctgs)} categories completed')

2min 28.5sec: pasting instruments and issuers for each of the 19 categories completed


In [70]:
# (5) prettify the schedule

start_time = time.time()
#print(f'Prettifying the schedule ...')

# static for formatting security values and percentages
nmbr = '#,##0.00 ;-#,##0.00 ;"- "' # https://support.microsoft.com/en-us/office/number-format-codes-5026bbd6-04bc-48cd-bf33-80f18b4eae68
fnt  = 'Calibri'
sz   = 12

# rows from top of schedule to the TOTAL row
row1 = item_row(sh, "Top"  , 7) # first row of interest in the schedule
rows = item_row(sh, "SAFEX", 7) # last row of interest in the schedule
for row in range(row1 - 1, rows):   #https://stackoverflow.com/questions/49525545/openpyxl-formatting-cell-with-decimal

    sh['D'][row].number_format = '0%'
    sh['D'][row].alignment     = Alignment(horizontal = 'center')
    sh['D'][row].alignment     = Alignment(vertical   = 'top')
    sh['D'][row].font          = Font(name = fnt)
    sh['D'][row].font          = Font(size = sz)
    if sh['D'][row].value == .025:
        sh['D'][row].number_format = '0.0%'
    
    sh['E'][row].number_format = nmbr
    sh['E'][row].alignment     = Alignment(horizontal = 'right')
    sh['E'][row].alignment     = Alignment(vertical   = 'top')
    sh['E'][row].font          = Font(name = fnt)
    sh['E'][row].font          = Font(size = sz)
    
    sh['F'][row].number_format = nmbr
    sh['F'][row].alignment     = Alignment(horizontal = 'right')
    sh['F'][row].alignment     = Alignment(vertical   = 'top')
    sh['F'][row].font          = Font(name = fnt)
    sh['F'][row].font          = Font(size = sz)

    sh['F8'].number_format     = nmbr   
    sh['F8'].alignment         = Alignment(horizontal = 'right')
    sh['F8'].alignment         = Alignment(vertical   = 'top')
    sh['F8'].font              = Font(name = fnt)
    sh['F8'].font              = Font(size = sz)

    sh['F13'].number_format    = nmbr
    sh['F13'].alignment        = Alignment(horizontal = 'right')
    sh['F13'].alignment        = Alignment(vertical   = 'top')
    sh['F13'].font             = Font(name = fnt)
    sh['F13'].font             = Font(size = sz)
    sh['F13'].font             = Font(bold = True)


# format the 'Limit (%)' column
# for row in range(row1 - 1, rows):
#     sh['D'][row].alignment     = Alignment(horizontal = 'center')
#     sh['D'][row].alignment     = Alignment(vertical   = 'top')
#     sh['D'][row].font          = Font(name = fnt)
#     sh['D'][row].font          = Font(size = sz)
#     if sh['D'][row].value != .025:
#         sh['D'][row].number_format = '0%'
#     if sh['D'][row].value == .025:
#         sh['D'][row].number_format = '0.0%'

# format the 'Limit (%)' column heading
sh['D15'].font = Font(bold = True)

# format the 'Fair Value' headings
cols = [4, 5]
for row in range(row1 - 1, rows):
    for col in cols:
        if 'Fair Value' in str(sh[row][col].value):           
            sh[row][col].font       = Font(bold = True)
            sh[row][col].alignment  = Alignment(horizontal = 'center')
            sh[row][col].alignment  = Alignment(vertical   = 'top')

# set column widths
widths = {'D': 8.57}
for col in widths:
    sh.column_dimensions[col].width = widths[col]

print(f'{timediff(start_time, time.time())}: prettifying the schedule completed')

3.1sec: prettifying the schedule completed


In [71]:
# (6) save the file

start_time = time.time()
pthTest     = r'P:\Working Folders\Hilton\W\Reg_Tests'     # where to save in Test folder
#print(f'Saving {fund} Reg28 SchIB {date.strftime("%d%b%Y")}.xlsx in {pthTest} ...')

wb.save(os.path.join(pthTest, f'{fund} Reg28 SchIB {date.strftime("%d%b%Y")}.xlsx')) # save the completed Schedule IB in the Test folder
wb.close

#open_xl_file(os.path.join(pthTest,f'{fund} Reg28 SchIB {date.strftime("%d%b%Y")}.xlsx'))

print(f'{timediff(start_time, time.time())}: saving {fund} Reg28 SchIB {date.strftime("%d%b%Y")}.xlsx in {pthTest} completed')

0.1sec: saving PIMBAL Reg28 SchIB 31Jul2024.xlsx in P:\Working Folders\Hilton\W\Reg_Tests completed


In [72]:
print('\n', f'{timediff(start_time0, time.time())}: roundtrip time for {len(funds)} fund{"" if len(funds) == 1 else "s"}')


 7.4sec: roundtrip time for 1 fund


In [73]:
cts   = {'1', '1.1', '1.1(a)', '1.1(b)', '1.1(c)', '1.1(d)', '1.2', '1.2(a)', '1.2(b)', '1.2(c)', '2', '2.1', '2.1(a)', '2.1(b)', '2.1(c)', \
         '2.1(d)', '2.1(e)', '2.2', '2.2(a)', '2.2(b)', '2.2(c)', '2.2(d)', '2.2(e)', '3', '3.1', '3.1(a)', '3.1(b)', '3.2(a)', '3.2(b)', \
         '4', '4.1', '4.1(a)', '4.1(b)', '4.2', '4.2(a)', '4.2(b)', '5', '5.1', '5.1(a)', '5.2', '5.2(a)', '6', '6(a)', '6(b)', '7', '8', \
         '8.1', '8.1(a)', '8.2', '8.2(a)', '9', '9.1', '9.1(a)', '9.2', '9.2(a)', '10', '10.1', '10.2'}

In [ ]:

# P:\Working Folders\Hilton\W\!Reg28Templates.xlsx
# P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting\UCTRFBAL Reg28 30Jun2024.xlsx
# P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting\UCTRFINC Reg28 30Jun2024.xlsx
# P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting\PIMBAL Reg28 31Jul2024.xlsx
# P:\Working Folders\Hilton\W\Reg_Tests\UCTRFBAL Reg28 SchIB 30Jun2024.xlsx
# P:\Working Folders\Hilton\W\Reg_Tests\UCTRFINC Reg28 SchIB 30Jun2024.xlsx
# P:\Working Folders\Hilton\W\Reg_Tests\PIMBAL Reg28 SchIB 31Jul2024.xlsx

In [3]:
# constants - supercats and superhens for use in function fghi()

# iterating through a nested dictionary - https://www.programiz.com/python-programming/nested-dictionary
supercats  = {
          '3(f)':
              {'3(f) 2.1(e)(ii)'  : ['2.1(e)(ii)'],
               '3(f) 3.1(b)'      : ['3.1(b)'],
               '3(f) 4.1(b)'      : ['4.1(b)'],
               '3(f) 8'           : ['8.1(a)(i)', '8.1(a)(ii)', '8.2(a)(i)', '8.2(a)(ii)'],
               '3(f) 9'           : ['9.1(a)(i)', '9.1(a)(ii)', '9.2(a)(i)', '9.2(a)(ii)'],
               '3(f) 10'          : ['10.1', '10.2']},

          '3(g)':
              {'3(g) 3.1(b)'      : ['3.1(b)'],
               '3(g) 9'           : ['9.1(a)(i)', '9.1(a)(ii)', '9.2(a)(i)', '9.2(a)(ii)']},
    
          '3(h)':
              {'3(h) 1.1'         : ['1.1(a)', '1.1(b)', '1.1(c)', '1.1(d)'],
               '3(h) 2.1(c)'      : ['2.1(c)(i)', '2.1(c)(ii)', '2.1(c)(iii)', '2.1(c)(iv)']},
    
          '3(i)':
              {'3(i) cash'        : ['1.2(a)', '1.2(b)', '1.2(c)'],
               '3(i) debt'        : ['2.1(b)', '2.2(a)', 
                                     '2.2(a)(i)', '2.2(a)(ii)', '2.2(a)(iii)', '2.2(a)(iv)',
                                     '2.2(b)(i)', '2.2(b)(ii)',
                                     '2.2(c)(i)', '2.2(c)(ii)', 
                                     '2.2(d)(i)', '2.2(d)(ii)', 
                                     '2.2(e)(i)', '2.2(e)(ii)'],
               '3(i) equity'      : ['3.2(a)(i)', '3.2(a)(ii)', '3.2(a)(iii)', '3.2(b)'],
               '3(i) property'    : ['4.2(a)(i)', '4.2(a)(ii)', '4.2(a)(iii)', '4.2(b)'],
               '3(i) commodities' : ['5.2(a)(i)', '5.2(a)(ii)'],
               '3(i) hpefs'       : ['8.2(a)(i)', '8.2(a)(ii)', '9.2(a)(i)', '9.2(a)(ii)', '10.2']}
         }

# superhens for single digit and triple digit sub-totals
superhens  = {'1':  {'1.1'  :  {'1.1(a)'  : {},
                                '1.1(b)'  : {},
                                '1.1(c)'  : {},
                                '1.1(d)'  : {}},
                     '1.2'  :  {'1.2(a)'  : {},
                                '1.2(b)'  : {},
                                '1.2(c)'  : {}}},
              '2':  {'2.1'  :  {'2.1(a)'  : {},
                                '2.1(b)'  : {},
                                '2.1(c)'  : {'2.1(c)(i)', '2.1(c)(ii)', '2.1(c)(iii)', '2.1(c)(iv)'},
                                '2.1(d)'  : {'2.1(d)(i)', '2.1(d)(ii)'},
                                '2.1(e)'  : {'2.1(e)(i)', '2.1(e)(ii)'}},
                     '2.2'  :  {'2.2(a)'  : {'2.2(a)(i)', '2.2(a)(ii)', '2.2(a)(iii)', '2.2(a)(iv)'},
                                '2.2(b)'  : {},
                                '2.2(c)'  : {'2.2(c)(i)', '2.2(c)(ii)', '2.2(c)(iii)', '2.2(c)(iv)'},
                                '2.2(d)'  : {'2.2(d)(i)', '2.2(d)(ii)'},
                                '2.2(e)'  : {'2.2(e)(i)', '2.2(e)(ii)'}}},
              '3':  {'3.1'  :  {'3.1(a)'  : {'3.1(a)(i)', '3.1(a)(ii)', '3.1(a)(iii)'},
                                '3.1(b)'  : {}},                   
                     '3.2'  :  {'3.2(a)'  : {'3.2(a)(i)', '3.2(a)(ii)', '3.2(a)(iii)'},
                                '3.2(b)'  : {}}},
              '4':  {'4.1'  :  {'4.1(a)'  : {'4.1(a)(i)', '4.1(a)(ii)', '4.1(a)(iii)'},
                                '4.1(b)'  : {}},         
                     '4.2'  :  {'4.2(a)'  : {'4.2(a)(i)', '4.2(a)(ii)', '4.2(a)(iii)'},
                                '4.2(b)'  : {}}},
              '5':  {'5.1'  :  {'5.1(a)'  : {'5.1(a)(i)', '5.1(a)(ii)'}},
                     '5.2'  :  {'5.2(a)'  : {'5.2(a)(i)', '5.2(a)(ii)'}}},
              '6':  {'6(a)' :  {}, 
                     '6(b)' :  {}},
              '7':  {},
              '8':  {'8.1'  :  {'8.1(a)'  : {'8.1(a)(i)', '8.1(a)(ii)'}},         
                     '8.2'  :  {'8.2(a)'  : {'8.2(a)(i)', '8.2(a)(ii)'}}},
              '9':  {'9.1'  :  {'9.1(a)'  : {'9.1(a)(i)', '9.1(a)(ii)'}},         
                     '9.2'  :  {'9.2(a)'  : {'9.2(a)(i)', '9.2(a)(ii)'}}},
              '10': {'10.1' :  {},
                     '10.2' :  {}},
             }

In [4]:
for superhen, values in superhens.items():
    print(superhen, values)

1 {'1.1': {'1.1(a)': {}, '1.1(b)': {}, '1.1(c)': {}, '1.1(d)': {}}, '1.2': {'1.2(a)': {}, '1.2(b)': {}, '1.2(c)': {}}}
2 {'2.1': {'2.1(a)': {}, '2.1(b)': {}, '2.1(c)': {'2.1(c)(i)', '2.1(c)(ii)', '2.1(c)(iv)', '2.1(c)(iii)'}, '2.1(d)': {'2.1(d)(ii)', '2.1(d)(i)'}, '2.1(e)': {'2.1(e)(i)', '2.1(e)(ii)'}}, '2.2': {'2.2(a)': {'2.2(a)(i)', '2.2(a)(ii)', '2.2(a)(iii)', '2.2(a)(iv)'}, '2.2(b)': {}, '2.2(c)': {'2.2(c)(iii)', '2.2(c)(ii)', '2.2(c)(i)', '2.2(c)(iv)'}, '2.2(d)': {'2.2(d)(i)', '2.2(d)(ii)'}, '2.2(e)': {'2.2(e)(i)', '2.2(e)(ii)'}}}
3 {'3.1': {'3.1(a)': {'3.1(a)(ii)', '3.1(a)(i)', '3.1(a)(iii)'}, '3.1(b)': {}}, '3.2': {'3.2(a)': {'3.2(a)(iii)', '3.2(a)(ii)', '3.2(a)(i)'}, '3.2(b)': {}}}
4 {'4.1': {'4.1(a)': {'4.1(a)(ii)', '4.1(a)(i)', '4.1(a)(iii)'}, '4.1(b)': {}}, '4.2': {'4.2(a)': {'4.2(a)(ii)', '4.2(a)(iii)', '4.2(a)(i)'}, '4.2(b)': {}}}
5 {'5.1': {'5.1(a)': {'5.1(a)(i)', '5.1(a)(ii)'}}, '5.2': {'5.2(a)': {'5.2(a)(i)', '5.2(a)(ii)'}}}
6 {'6(a)': {}, '6(b)': {}}
7 {}
8 {'8.1': {'8

In [7]:
for cat_1, values_1 in superhens.items(): # for key, value in dict.items(key, values)
    print('\nLevel 1 Category:', cat_1)
    for cat_2, values_2 in values_1.items():
        print('\tLevel 2 Category:', cat_2)
        for cat_3, values_3 in values_2.items():
            print('\t\tAlphabetic category:', cat_3)
            for set_item in values_3:
                print('\t\t\tRoman category:', set_item)


Level 1 Category: 1
	Level 2 Category: 1.1
		Alphabetic category: 1.1(a)
		Alphabetic category: 1.1(b)
		Alphabetic category: 1.1(c)
		Alphabetic category: 1.1(d)
	Level 2 Category: 1.2
		Alphabetic category: 1.2(a)
		Alphabetic category: 1.2(b)
		Alphabetic category: 1.2(c)

Level 1 Category: 2
	Level 2 Category: 2.1
		Alphabetic category: 2.1(a)
		Alphabetic category: 2.1(b)
		Alphabetic category: 2.1(c)
			Roman category: 2.1(c)(i)
			Roman category: 2.1(c)(ii)
			Roman category: 2.1(c)(iv)
			Roman category: 2.1(c)(iii)
		Alphabetic category: 2.1(d)
			Roman category: 2.1(d)(ii)
			Roman category: 2.1(d)(i)
		Alphabetic category: 2.1(e)
			Roman category: 2.1(e)(i)
			Roman category: 2.1(e)(ii)
	Level 2 Category: 2.2
		Alphabetic category: 2.2(a)
			Roman category: 2.2(a)(i)
			Roman category: 2.2(a)(ii)
			Roman category: 2.2(a)(iii)
			Roman category: 2.2(a)(iv)
		Alphabetic category: 2.2(b)
		Alphabetic category: 2.2(c)
			Roman category: 2.2(c)(iii)
			Roman category: 2.2(c)(i

In [ ]:
# # iterating through a nested dictionary - https://www.programiz.com/python-programming/nested-dictionary
# for ctg, values in ctgs.items():                       # for key, associated item in ctgs dictionary
#     print('\nMain category:', ctg)                          # access only each category key, i.e.,  3(f), 3(g), 3(h), and 3(i)
    
#     for schd_row, lists in values.items():                            # for key,associated item in ctg dictionary
#         print('\t' + schd_row + ' contains this list -', values[schd_row], ', the elements of which are:') # access the row item

#         for cat in lists:
#             print('\t\t' + schd_row, cat)

In [ ]:
# # iterating through a nested dictionary - https://www.programiz.com/python-programming/nested-dictionary
# for ctg, values in ctgs.items():                       # for key, associated item in ctgs dictionary ...
#     #print('\nCategory:', ctg + ' -')                   # access only each category key, i.e.,  3(f), 3(g), 3(h), and 3(i)
    
#     for schd_row, lists in values.items():             # for key, associated item in ctg dictionary ...
#         #print('\t' + schd_row + ':')                   # access the row item

#         # create an empty dataframe - https://stackoverflow.com/questions/44513738/pandas-create-empty-dataframe-with-only-column-names
#         # https://stackoverflow.com/questions/19482970/get-a-list-from-pandas-dataframe-column-headers
#         k = pd.DataFrame(columns = r28.columns.values.tolist()) # empty dataframe with r28 headings
#         for cat in lists:
#             #print('\t\t' + cat)
#             #j = r28[r28['R28C'] == cat]
#             k = pd.concat(r28[r28['R28C'] == cat])
#             k.reset_index(drop = True, inplace = True) # re-index in place + drop the newly inserted index

#         sh.insert_rows(item_row(sh, schd_row, 7), len(k))       # insert as many rows as length of dataframe k at the row item

#         for index, row in k.iterrows():                     # loop over each security
#             rw = row_item + 1 + index                  # iterate sequentially over the empty rows, pasting securities along the way ...
#             sh[rw][2].value  = row['Instr']
#             sh[rw][4].value  = row['EMV']
#             sh[rw][5].value  = row['PMV']
#             sh[rw][4].border = Border(left  = Side(style = 'thin'))
#             sh[rw][5].border = Border(right = Side(style = 'thin'))
#             # sh[rw + len()][4].border = Border(top    = Side(style = 'thin'))
#             # sh[rw + len()][5].border = Border(top    = Side(style = 'thin'))
#             # sh[rw + len()][4].border = Border(bottom = Side(style = 'thin'))
#             # sh[rw + len()][5].border = Border(bottom = Side(style = 'thin'))
#             #sh.row_dimensions[rw].height = ht # https://stackoverflow.com/questions/37891149/openpyxl-auto-height-row